In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("Number of GPUs:", torch.cuda.device_count())

CUDA available: True
GPU name: Tesla T4
Number of GPUs: 1


Installations

In [32]:
!pip install requests tqdm

cell 2 - XenoCanto API 

In [33]:
import os

# Xeno-canto API key (keep this private)
os.environ["XENOCANTO_API_KEY"] = "66a66df58dca4ebd5610998429c48ad70d89ee5b"


Selection of species

In [ ]:
TARGET_SPECIES = [
    "Corvus splendens",
    "Acridotheres tristis",
    "Passer domesticus",
    "Columba livia",
    "Psittacula krameri"
]

DATASET_DIR = "/content/drive/MyDrive/bird_audio_trial"

MIN_DURATION = 2
MAX_DURATION = 10
ALLOWED_QUALITY = ["A", "B"]
MAX_FILES_PER_SPECIES = 50


Fetching of audio recodings from xenocanto

In [7]:
import requests
import os

API_KEY = os.environ.get("XENOCANTO_API_KEY")

def fetch_xenocanto_page(species_name, page):
    genus, species = species_name.split()
    query = f"gen:{genus} sp:{species}"

    url = "https://xeno-canto.org/api/3/recordings"
    params = {
        "query": query,
        "key": API_KEY,
        "per_page": 100,
        "page": page
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    return response.json()


Testing the successful retrival of audio recordings

In [8]:
test = fetch_xenocanto_page("Corvus splendens", 1)

print("Recordings:", test["numRecordings"])
print("Pages:", test["numPages"])
print("First ID:", test["recordings"][0]["id"])
print("File URL:", test["recordings"][0]["file"])

Recordings: 219
Pages: 3
First ID: 1072923
File URL: https://xeno-canto.org/1072923/download


checks the recording is valid or not

In [9]:
def is_valid_recording(record):
    try:
        mins, secs = record["length"].split(":")
        duration = int(mins) * 60 + int(secs)

        return (
            MIN_DURATION <= duration <= MAX_DURATION
            and record["q"] in ALLOWED_QUALITY
        )
    except:
        return False


downloading function

In [10]:
import os
from tqdm import tqdm

def download_species_audio(species_name):
    species_folder = species_name.replace(" ", "_")
    save_path = os.path.join(DATASET_DIR, species_folder)
    os.makedirs(save_path, exist_ok=True)

    downloaded = 0
    page = 1

    while downloaded < MAX_FILES_PER_SPECIES:
        data = fetch_xenocanto_page(species_name, page)

        for record in data["recordings"]:
            if downloaded >= MAX_FILES_PER_SPECIES:
                break

            if not is_valid_recording(record):
                continue

            # ✅ CORRECT URL HANDLING
            file_url = record["file"]
            if file_url.startswith("//"):
                audio_url = "https:" + file_url
            elif file_url.startswith("http"):
                audio_url = file_url
            else:
                continue

            file_name = f"XC{record['id']}.mp3"
            file_path = os.path.join(save_path, file_name)

            if os.path.exists(file_path):
                continue

            r = requests.get(audio_url, stream=True)
            with open(file_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024):
                    if chunk:
                        f.write(chunk)

            downloaded += 1

        if page >= int(data["numPages"]):
            break
        page += 1

    print(f"✅ {species_name}: {downloaded} files downloaded")


In [11]:
for species in TARGET_SPECIES:
    print(f"\nDownloading trial data for: {species}")
    download_species_audio(species)



✅ Corvus splendens: 0 files downloaded

✅ Acridotheres tristis: 20 files downloaded

✅ Passer domesticus: 20 files downloaded

✅ Columba livia: 0 files downloaded

✅ Psittacula krameri: 20 files downloaded


cell 0


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


New structure

cell 1

In [4]:
!pip install requests tqdm

cell 2

In [5]:
import os

# Xeno-canto API key (keep this private)
os.environ["XENOCANTO_API_KEY"] = "66a66df58dca4ebd5610998429c48ad70d89ee5b"


cell 3

In [6]:
import os

# Base project directory
BASE_DIR = "/content/drive/MyDrive/bird_audio_detector"

# All downloaded audio will go inside this folder
DATASET_DIR = os.path.join(BASE_DIR, "input")

# Create input folder if it doesn't exist
os.makedirs(DATASET_DIR, exist_ok=True)

TARGET_SPECIES = [
    "Corvus splendens",
    "Acridotheres tristis",
    "Passer domesticus",
    "Columba livia",
    "Psittacula krameri"
]

MIN_DURATION = 2
MAX_DURATION = 10
ALLOWED_QUALITY = ["A", "B"]
MAX_FILES_PER_SPECIES = 50

cell 4

In [7]:
import requests

API_KEY = os.environ.get("XENOCANTO_API_KEY")

def fetch_xenocanto_page(species_name, page):
    genus, species = species_name.split()
    query = f"gen:{genus} sp:{species}"

    url = "https://xeno-canto.org/api/3/recordings"
    params = {
        "query": query,
        "key": API_KEY,
        "per_page": 100,
        "page": page
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    return response.json()

cell 5

In [8]:
test = fetch_xenocanto_page("Corvus splendens", 1)

print("Recordings:", test["numRecordings"])
print("Pages:", test["numPages"])
print("First ID:", test["recordings"][0]["id"])
print("File URL:", test["recordings"][0]["file"])

Recordings: 219
Pages: 3
First ID: 1072923
File URL: https://xeno-canto.org/1072923/download


cell 6

In [9]:
def is_valid_recording(record):
    try:
        mins, secs = record["length"].split(":")
        duration = int(mins) * 60 + int(secs)

        return (
            MIN_DURATION <= duration <= MAX_DURATION
            and record["q"] in ALLOWED_QUALITY
        )
    except:
        return False

cell 7

In [10]:
from tqdm import tqdm

def download_species_audio(species_name):
    species_folder = species_name.replace(" ", "_")
    save_path = os.path.join(DATASET_DIR, species_folder)
    os.makedirs(save_path, exist_ok=True)

    downloaded = 0
    page = 1

    while downloaded < MAX_FILES_PER_SPECIES:
        data = fetch_xenocanto_page(species_name, page)

        for record in data["recordings"]:
            if downloaded >= MAX_FILES_PER_SPECIES:
                break

            if not is_valid_recording(record):
                continue

            file_url = record["file"]

            if file_url.startswith("//"):
                audio_url = "https:" + file_url
            elif file_url.startswith("http"):
                audio_url = file_url
            else:
                continue

            file_name = f"XC{record['id']}.mp3"
            file_path = os.path.join(save_path, file_name)

            if os.path.exists(file_path):
                continue

            try:
                r = requests.get(audio_url, stream=True, timeout=30)
                r.raise_for_status()

                with open(file_path, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024):
                        if chunk:
                            f.write(chunk)

                downloaded += 1

            except Exception as e:
                print(f"Failed to download {audio_url}: {e}")
                continue

        if page >= int(data["numPages"]):
            break

        page += 1

    print(f"✅ {species_name}: {downloaded} files downloaded")

cell 8

In [11]:
for species in TARGET_SPECIES:
    print(f"\nDownloading trial data for: {species}")
    download_species_audio(species)


✅ Corvus splendens: 10 files downloaded

✅ Acridotheres tristis: 42 files downloaded

✅ Passer domesticus: 50 files downloaded

✅ Columba livia: 15 files downloaded

✅ Psittacula krameri: 50 files downloaded


In [6]:
%cd /content/drive/MyDrive/bird_audio_detector

/content/drive/MyDrive/bird_audio_detector


In [7]:
!pip install librosa numpy soundfile tqdm scikit-learn


In [8]:
!python main.py

Initializing Math-based NMF Separator (components=2)...
Starting dataset preprocessing...
Corvus_splendens - train: 100% 21/21 [00:35<00:00,  1.71s/it]
Corvus_splendens - val:  50% 2/4 [00:01<00:01,  1.54it/s]/usr/local/lib/python3.12/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
Corvus_splendens - val: 100% 4/4 [00:04<00:00,  1.13s/it]
Corvus_splendens - test:  20% 1/5 [00:00<00:02,  1.57it/s]/usr/local/lib/python3.12/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(
Corvus_splendens - test:  60% 3/5 [00:02<00:01,  1.04it/s]Warning: Xing stream size off by more than 1%, fuzzy seeking may be even more fuzzy than by design!
Corvus_splendens - test: 100% 5/5 [00:03<00:00,  1.32it/s]
Acridotheres_tristis - train:  26% 11/43 [00:05<00:14,  2.16it/s]Warning: Xi

Test codes

In [ ]:
import numpy as np

data = np.load("XC79536_source_0.npy")
print(data)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import librosa.display

# Load the two separated sources of a single example file
file_0 = "processed/train/Corvus_splendens/XC165854_source_0.npy" # change to a real filename you have
file_1 = "processed/train/Corvus_splendens/XC165854_source_1.npy"

mel_0 = np.load(file_0)
mel_1 = np.load(file_1)

print(f"Shape of Source 0: {mel_0.shape}")
print(f"Value Range: Min {mel_0.min():.2f}, Max {mel_0.max():.2f}")

# Visualize them side-by-side
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
librosa.display.specshow(mel_0, x_axis='time', y_axis='mel', sr=22050)
plt.title('Source 0 (e.g., Bird / Noise)')
plt.colorbar(format='%+2.0f dB')

plt.subplot(1, 2, 2)
librosa.display.specshow(mel_1, x_axis='time', y_axis='mel', sr=22050)
plt.title('Source 1 (e.g., Background / Other Bird)')
plt.colorbar(format='%+2.0f dB')
plt.tight_layout()
plt.show()
